# PART 02 · 2장 작업형 1 핵심 유형 — 연습 문제 정답

교재 190쪽 연습 문제의 정답 코드입니다. 노트북과 같은 위치에 `data` 폴더를 두고 위에서부터 차례대로 실행하세요.

## 공통 준비

In [4]:
import pandas as pd

missing_df = pd.read_csv('/content/sample_data/type1_missing_outlier.csv')
date_df = pd.read_csv('/content/sample_data/type1_date_string.csv')
print(missing_df.shape, date_df.shape)

(350, 8) (400, 7)


## 문제 01

item_group별 views 중앙값으로 결측치를 채우시오. views를 qty로 나눈 views_per_qty를 만든 뒤 views_per_qty 내림차순, row_id 오름차순으로 정렬했을 때 세 번째 행의 row_id를 구하시오.

**정답 코드**

In [5]:
result = missing_df.copy()
group_median = (
    result.groupby('item_group')['views']
    .transform('median')
)
result['views'] = result['views'].fillna(group_median)
result['views_per_qty'] = result['views'] / result['qty']
result = result.sort_values(
    ['views_per_qty', 'row_id'],
    ascending=[False, True]
)
answer = result.iloc[2]['row_id']
print(answer)

MO0196


**간단 해설**

item_group별 views 중앙값을 원본과 같은 길이와 인덱스로 만든 뒤 결측치를 채운다. 이 데이터의 qty는 모두 1 이상이므로 0으로 나누는 경우는 없다. views_per_qty를 만들고 두 정렬 기준을 차례대로 적용한다. 정렬 결과의 세 번째 위치는 iloc[2]이며 row_id는 MO0196이다.

## 문제 02

reserved_at을 날짜형으로 변환하고, service_code의 첫 번째 조각으로 service_type을 만드시오. 예약 시간을 새벽·오전·오후·저녁으로 구분한 뒤, service_type이 BOOK 또는 CARE이고 status가 방문인 행에서 건수가 가장 많은 시간대의 이름을 구하시오.

**정답 코드**

In [6]:
result = date_df.copy()
result['reserved_at'] = pd.to_datetime(result['reserved_at'])
result['service_type'] = (
    result['service_code'].str.split('-').str[0]
)
result['reserved_hour'] = result['reserved_at'].dt.hour
result['time_group'] = pd.cut(
    result['reserved_hour'],
    bins=[0, 6, 12, 18, 24],
    labels=['새벽', '오전', '오후', '저녁'],
    right=False
)
cond = (
    result['service_type'].isin(['BOOK', 'CARE'])
    & (result['status'] == '방문')
)
time_counts = (
    result.loc[cond]
    .groupby('time_group', observed=False)
    .size()
)
answer = time_counts.idxmax()
print(answer)

저녁


In [7]:
time_counts

,0
time_group,
새벽,0
오전,24
오후,24
저녁,29


In [10]:
result['time_group'].value_counts()

,count
time_group,
오후,148
저녁,128
오전,124
새벽,0


**간단 해설**

service_type에서 서비스 종류를 추출하고 reserved_at의 시간으로 시간대 구간을 만든다. BOOK 또는 CARE이면서 방문 상태인 행만 선택한 뒤 시간대별 건수를 계산한다. 가장 많은 시간대는 저녁이다.

---

**정답** 1. MO0196, 2. 저녁